# Bike Taxi Demand Prediction

This notebook explores the dataset, prepares features, compares regression models and evaluates ride-demand predictions.

Place the dataset at `data/raw/ola.csv` before running the notebook.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from bike_demand.data import load_dataset
from bike_demand.features import prepare_dataset
from bike_demand.train import train_and_select_model

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "ola.csv"
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}. See data/README.md for the required schema and attribution guidance."
    )

raw = load_dataset(DATA_PATH)
print(f"Rows: {len(raw):,}")
print(f"Columns: {len(raw.columns)}")
display(raw.head())

## 1. Dataset quality and schema

In [ ]:
quality = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing_values": raw.isna().sum(),
    "missing_percent": (raw.isna().mean() * 100).round(2),
    "unique_values": raw.nunique(dropna=True),
})
display(quality)
display(raw.describe(include="all").T)

## 2. Feature preparation

In [ ]:
prepared = prepare_dataset(raw, training=True)
model_frame = prepared.frame
print("Prepared shape:", model_frame.shape)
print("Predictor columns:", prepared.feature_columns)
print("Target:", prepared.target_column)
display(model_frame.head())

## 3. Exploratory analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
model_frame["count"].plot(kind="hist", bins=30, ax=axes[0], title="Ride demand distribution")
axes[0].set_xlabel("Ride count")
hourly = model_frame.groupby("hour")["count"].mean()
hourly.plot(kind="bar", ax=axes[1], title="Average demand by hour")
axes[1].set_xlabel("Hour of day")
axes[1].set_ylabel("Average ride count")
plt.tight_layout()
plt.show()

daily = model_frame.set_index("datetime")["count"].resample("D").sum()
plt.figure(figsize=(14, 4))
daily.plot()
plt.title("Daily total ride demand")
plt.xlabel("Date")
plt.ylabel("Total ride count")
plt.tight_layout()
plt.show()

## 4. Chronological model selection and holdout evaluation

In [ ]:
result = train_and_select_model(
    data_path=DATA_PATH,
    artifact_path=PROJECT_ROOT / "artifacts" / "model_bundle.joblib",
    leaderboard_path=PROJECT_ROOT / "artifacts" / "model_leaderboard.csv",
    predictions_path=PROJECT_ROOT / "artifacts" / "test_predictions.csv",
    feature_importance_path=PROJECT_ROOT / "artifacts" / "feature_importance.csv",
)
print("Selected model:", result.best_model_name)
print("Untouched chronological holdout metrics:")
display(pd.Series(result.test_metrics, name="value"))
display(result.leaderboard)

## 5. Prediction diagnostics

In [ ]:
predictions = pd.read_csv(PROJECT_ROOT / "artifacts" / "test_predictions.csv")
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(predictions["count"], predictions["predicted_count"], alpha=0.7)
limit = max(predictions["count"].max(), predictions["predicted_count"].max())
axes[0].plot([0, limit], [0, limit], linestyle="--", color="black")
axes[0].set(title="Actual vs predicted", xlabel="Actual ride count", ylabel="Predicted ride count")
axes[1].hist(predictions["residual"], bins=25)
axes[1].set(title="Residual distribution", xlabel="Actual - predicted", ylabel="Frequency")
plt.tight_layout()
plt.show()

## 6. Interpretation and limitations

- Models are compared using validation MAE.
- The final chronological test set is kept separate until model selection is complete.
- Results depend on the dataset version and time period.
- The dataset is not included in the repository; use an authorized copy at `data/raw/ola.csv`.
- The current workflow does not use lagged or rolling demand features.
